# NB2h — Salvage every carry-over + extend Sonnet's share (Batch API)

Two jobs in one batch, because I have Anthropic credit that can only be spent on Claude:

1. **Salvage every unsalvaged reject** from all four finished generators — DeepSeek, Qwen, Sonnet
   and Opus (130 cards: 0 + 11 + 29 + 90; DeepSeek's original 22 were already rescued by Qwen,
   which is the relay working exactly as designed).
2. **Take {N_FRESH} fresh cards** off the front of the untouched pool, drawn with the same moving
   pointer every generator uses, so no card is ever processed twice.

**Why Sonnet for both?** It passed 93% vs Opus's 81% on the identical gate and costs ~1/3 as much
per article — the stronger *and* cheaper salvager. Its share of the corpus grows as a result, which
I'm accepting deliberately (see notes).

**Input:** `fact_cards.parquet` + all four prior outputs.
**Output:** `aig_sonnet_retry.parquet`.

## What I changed, and why

Both models failed the gate almost entirely on **number coverage** (Sonnet rejects: 45%, Opus
rejects: 30%) while entities (89-93%) and agencies (90-92%) were fine. They paraphrase figures
("عشرات المهاجرين") instead of carrying them over verbatim. So the one change here is the same
light number emphasis that lifted Qwen's gate from 76% to 93%:

    'أدرج هذه الأرقام والتواريخ: ...'          ->  before
    'احرص على ذكر هذه الأرقام والتواريخ كما هي: ...'  ->  now

This is **factual fidelity** (does the article carry its card's facts?), not tuning against any of
my five detection features. Everything else — system prompt, style hints, gate, thresholds — is
byte-identical to NB2f/NB2g so the AI class stays consistently defined.

## Lessons already baked in

- **No `temperature` / `top_p` / `top_k`.** Sonnet 5 and Opus 4.8 reject them outright
  (400: "`temperature` is deprecated for this model"). Diversity comes from the prompt: each card
  samples 3-4 different style hints and carries different content.
- **`thinking: disabled`.** Sonnet 5 runs adaptive thinking by default and those tokens eat into
  `max_tokens` — which would silently truncate long reports.
- **Generous `max_tokens`.** Arabic runs ~3.5 tok/word and Claude's tokenizer adds ~30%, so I size
  at 5.5x target words. It's a ceiling, not a charge.
- **Smoke test before submitting.** One synchronous call validates the params; it's what caught the
  temperature problem before it cost a second failed batch.
- **Real error messages** are printed on collect, so any failure is diagnosable immediately.

## Config — BATCH_ID empty = submit (run 1), filled = collect (run 2)

In [1]:
!pip -q install anthropic >/dev/null 2>&1

import pandas as pd, numpy as np, json, re, os, time, random, glob
import anthropic
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request
from kaggle_secrets import UserSecretsClient

API_KEY = UserSecretsClient().get_secret('ANTHROPIC_API_KEY')
client  = anthropic.Anthropic(api_key=API_KEY)

GEN_ID     = 'sonnet'
MODEL_NAME = 'claude-sonnet-5'
PRICE_IN, PRICE_OUT = 2.0, 10.0

# How many FRESH (never-processed) cards to add on top of the carry-over.
# Measured cost from my first Sonnet batch: $6.45 / 470 = ~$0.0137 per article.
#   200 fresh -> 330 total -> ~$4.5    (keeps Sonnet ~21% of the corpus)
#   350 fresh -> 480 total -> ~$6.6    (Sonnet ~25%, on par with DeepSeek — my pick)
#   598 fresh -> 728 total -> ~$10.0   (Sonnet ~31%, becomes the dominant generator)
N_FRESH = 350

CARDS_PATH = '/kaggle/input/notebooks/bahaaqassem/nb2c-build-fact-cards/fact_cards.parquet'
OUT_DIR    = '/kaggle/working'
BATCH_META = f'{OUT_DIR}/batch_meta_retry.json'

# Outputs whose rejects I'm salvaging. A card counts as "still unsalvaged" only if it failed
# somewhere and passed nowhere.
RETRY_SOURCES = [
    '/kaggle/input/notebooks/bahaaqassem/nb2d-generate-deepseek/aig_deepseek.parquet',
    '/kaggle/input/datasets/bahaaqassem/aig-qwen/aig_qwen.parquet',
    '/kaggle/input/notebooks/bahaaqassem/nb2f-generate-sonnet/aig_sonnet.parquet',
    '/kaggle/input/notebooks/bahaaqassem/nb2g-generate-opus/aig_opus.parquet',
]

# ================== THE ONE SWITCH ==================
# Run 1 (submit): leave empty.  Run 2 (collect): paste the msgbatch_... id printed by run 1.
BATCH_ID = 'msgbatch_013jEiEvWwxvPaCWSqJB7aGv'
# ===================================================

def find_parquet(preferred, *keywords):
    if os.path.exists(preferred):
        return preferred
    for kw in keywords:
        hits = [p for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})'); return hits[0]
    print('AVAILABLE /kaggle/input parquet files:')
    for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True): print('   ', p)
    raise FileNotFoundError(preferred)

CARDS_PATH = find_parquet(CARDS_PATH, 'fact_cards', 'fact-cards')
cards = pd.read_parquet(CARDS_PATH)
print('cards:', cards.shape, '| mode:', 'COLLECT (run 2)' if BATCH_ID else 'SUBMIT (run 1)')

cards: (3500, 16) | mode: COLLECT (run 2)


## Prompt, gate, normalization — identical to NB2f/NB2g except the number line

In [2]:
STYLE_HINTS = [
    'ابدأ بفقرة استهلالية تلخّص الحدث', 'اذكر خلفية موجزة للسياق',
    'أورد ردود فعل الأطراف المعنية', 'اختم بما هو متوقّع أو منتظر',
    'استخدم بنية الهرم المقلوب', 'انسب المعلومات إلى مصادرها',
]
SYSTEM_PROMPT = (
    'أنت صحفي محترف يكتب تقارير إخبارية بالعربية الفصحى لصالح غرفة أخبار محترمة. '
    'تكتب بأسلوب صحفي رصين ودقيق، وتلتزم بالحقائق المعطاة دون إضافة معلومات من خارجها.'
)
MONTH_GROUPS = [
    {'يناير','كانون الثاني'}, {'فبراير','شباط'}, {'مارس','آذار'}, {'أبريل','نيسان'},
    {'مايو','أيار'}, {'يونيو','حزيران'}, {'يوليو','تموز'}, {'أغسطس','آب'},
    {'سبتمبر','أيلول'}, {'أكتوبر','تشرين الأول'}, {'نوفمبر','تشرين الثاني'}, {'ديسمبر','كانون الأول'},
]
AMBIG = {'كانون': 0, 'تشرين': 9}

def month_group(item):
    for i, g in enumerate(MONTH_GROUPS):
        if any(m in item for m in g): return i
    for k, i in AMBIG.items():
        if k in item: return i
    return None

def clean_numbers(nums, cap=6):
    seen, out = set(), []
    for n in nums:
        n = n.strip()
        if re.fullmatch(r'[٠-٩0-9]', n): continue
        g = month_group(n)
        if g is not None:
            if g in seen: continue
            seen.add(g)
        out.append(n)
    return out[:cap]

def build_prompt(card):
    ents  = json.loads(card['entities_for_prompt'])
    facts = json.loads(card['fact_points'])
    target = int(card['target_words'])
    parts = ['اكتب تقريراً إخبارياً بالعربية الفصحى عن الموضوع التالي.', '',
             f'الموضوع: {card["topic_core"]}', '',
             'الكيانات التي يجب أن يذكرها التقرير:', '، '.join(ents), '',
             'الحقائق الأساسية التي يجب تغطيتها:']
    for f in facts: parts.append(f'- {f}')
    if card['has_numbers']:
        nums = clean_numbers(json.loads(card['numbers_dates']))
        if nums:
            # THE ONE CHANGE vs NB2f/NB2g: both models dropped figures, so I ask once, clearly,
            # for the numbers verbatim. Same wording that lifted Qwen's gate 76% -> 93%.
            parts += ['', 'احرص على ذكر هذه الأرقام والتواريخ كما هي: ' + '، '.join(nums)]
    if card['has_agencies']:
        parts += ['', 'انسب المعلومات إلى: ' + '، '.join(json.loads(card['source_agencies']))]
    if card['has_quotes']:
        parts += ['', 'أدرج تصريحات منسوبة للأطراف المعنية، بصياغتك أنت.']
    hints = random.sample(STYLE_HINTS, k=random.choice([3, 4]))
    parts += ['', 'إرشادات التحرير:'] + [f'- {h}' for h in hints]
    parts += ['', f'الطول: لا يقل التقرير عن {target} كلمة ولا يزيد عن {int(target*1.12)} كلمة. '
                  f'اكتب تقريراً مكتملاً ضمن هذا النطاق.', '',
              'اكتب نص التقرير مباشرة: دون عنوان، ودون أي تنسيق (لا نجوم ** ولا رموز تنسيق)، '
              'ودون مقدمة أو تعليق منك.']
    return '\n'.join(parts)

def normalize_format(text):
    t = str(text)
    t = re.sub(r'\*\*(.+?)\*\*', r'\1', t)
    t = re.sub(r'__(.+?)__', r'\1', t)
    t = re.sub(r'(?<!\w)\*(.+?)\*(?!\w)', r'\1', t)
    t = re.sub(r'^#{1,6}\s*', '', t, flags=re.M)
    t = re.sub(r'^\s*[-–—>]\s+', '', t, flags=re.M)
    t = re.sub(r'^\s*[-*_]{3,}\s*$', '', t, flags=re.M)
    t = re.sub(r'\n+', ' ', t)
    t = re.sub(r'\s{2,}', ' ', t)
    return t.strip()

_AR_DIGITS = str.maketrans('٠١٢٣٤٥٦٧٨٩',
                           '0123456789')
def _norm(s):
    s = re.sub(r'[\u064B-\u0652]', '', s); s = s.translate(_AR_DIGITS)
    return (s.replace('أ','ا').replace('إ','ا').replace('آ','ا')
             .replace('ة','ه').replace('ى','ي'))
def _norm_entity(e):
    e = _norm(e)
    e = re.sub(r'^[وفبكل]?ال', '', e)
    e = re.sub(r'^لل', '', e)
    e = re.sub(r'^[وفبكل](?=.{3,})', '', e)
    return e.strip()
def _present(item, tn, is_entity=True):
    n = _norm_entity(item) if is_entity else _norm(item)
    return len(n) >= 2 and n in tn
def _number_present(item, tn):
    g = month_group(item)
    if g is not None:
        names = MONTH_GROUPS[g] | {k for k, v in AMBIG.items() if v == g}
        return any(_norm(m) in tn for m in names)
    return _present(item, tn, is_entity=False)

def acceptance_gate(article, card, W_ENT=3, W_NUM=2, W_AG=1):
    tn = _norm(article); scores, weights, detail = [], [], {}
    ents = json.loads(card['entities_for_prompt'])
    if ents:
        hit = sum(_present(e, tn) for e in ents); ent_cov = hit/len(ents)
        detail['entities'] = f'{hit}/{len(ents)}'; scores.append(ent_cov); weights.append(W_ENT)
    else:
        ent_cov = 1.0; detail['entities'] = 'none'
    if card['has_numbers']:
        nums = clean_numbers(json.loads(card['numbers_dates']))
        if nums:
            hit = sum(_number_present(n, tn) for n in nums)
            scores.append(hit/len(nums)); weights.append(W_NUM); detail['numbers'] = f'{hit}/{len(nums)}'
    if card['has_agencies']:
        ags = json.loads(card['source_agencies']); hit = sum(_present(a, tn, False) for a in ags)
        scores.append(hit/max(len(ags),1)); weights.append(W_AG); detail['agencies'] = f'{hit}/{len(ags)}'
    weighted = sum(s*w for s, w in zip(scores, weights)) / max(sum(weights), 1)
    passed = (weighted >= 0.80) and (ent_cov >= 0.65)
    return {'passed': passed, 'weighted': round(weighted, 3),
            'entity_cov': round(ent_cov, 3), 'detail': detail}

def length_ok(article, target, tol=0.15):
    n = len(article.split()); return abs(n - target)/target <= tol, n

## Work list = carry-over + fresh cards

A card enters the **carry-over** only if it failed somewhere and passed nowhere (`failed -
succeeded`). That subtraction is what makes the relay self-correcting: DeepSeek's 22 rejects don't
appear here because Qwen already rescued all of them.

The **fresh** cards are the first `N_FRESH` of the ordered list that no generator has touched — the
same moving pointer every notebook uses, so Gemini and GPT simply start further along.

In [3]:
succeeded, failed, processed = set(), set(), set()
for path in RETRY_SOURCES:
    if not os.path.exists(path):
        kw = os.path.basename(path).replace('.parquet', '')
        hits = [p for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True)
                if kw in p or kw.replace('_', '-') in p]
        path = hits[0] if hits else path
    if os.path.exists(path):
        prev = pd.read_parquet(path)
        succeeded |= set(prev.loc[prev['gate_passed'], 'source_pair_id'])
        failed    |= set(prev.loc[~prev['gate_passed'], 'source_pair_id'])
        processed |= set(prev['source_pair_id'])
        n_bad = int((~prev['gate_passed']).sum())
        print(f'{os.path.basename(path)}: {len(prev)} rows, {n_bad} rejects')
    else:
        raise FileNotFoundError(f'retry source missing: {path}')

carry_over  = failed - succeeded                       # failed somewhere, rescued nowhere
carry_cards = cards[cards['pair_id'].isin(carry_over)]

# fresh cards: first N_FRESH never-processed cards, in the one canonical order
ordered     = cards.sample(frac=1.0, random_state=42).reset_index(drop=True)
fresh_cards = ordered[~ordered['pair_id'].isin(processed)].head(N_FRESH)

my_cards = (pd.concat([carry_cards, fresh_cards])
              .drop_duplicates('pair_id')
              .reset_index(drop=True))

est = len(my_cards) * 6.45 / 470                       # measured $/article from my first batch
print(f'\ncarry-over (unsalvaged): {len(carry_cards)}')
print(f'fresh cards           : {len(fresh_cards)}  (of {(~ordered["pair_id"].isin(processed)).sum()} untouched)')
print(f'TOTAL to send         : {len(my_cards)}  | duplicates: {my_cards["pair_id"].duplicated().sum()}')
print(f'estimated cost        : ~${est:.2f}')
print('sample ids:', my_cards['pair_id'].head(3).tolist())

aig_deepseek.parquet: 900 rows, 22 rejects
aig_qwen.parquet: 529 rows, 11 rejects
aig_sonnet.parquet: 470 rows, 34 rejects
aig_opus.parquet: 470 rows, 90 rejects

carry-over (unsalvaged): 135
fresh cards           : 350  (of 1153 untouched)
TOTAL to send         : 485  | duplicates: 0
estimated cost        : ~$6.66
sample ids: ['HA_00072', 'HA_00100', 'HA_00135']


## Smoke test — one synchronous call before spending on the batch

This is what caught the deprecated-`temperature` error last time. One live call, a fraction of a
cent, and I know the params are valid before submitting anything.

In [4]:
if not BATCH_ID:
    _c = my_cards.iloc[0]
    try:
        _m = client.messages.create(
            model=MODEL_NAME, max_tokens=600, system=SYSTEM_PROMPT,
            messages=[{'role': 'user', 'content': build_prompt(_c)}],
            thinking={'type': 'disabled'},
        )
        _txt = ''.join(b.text for b in _m.content if b.type == 'text')
        print('SMOKE TEST OK — params accepted by', MODEL_NAME)
        print('  tokens:', _m.usage.input_tokens, 'in /', _m.usage.output_tokens, 'out')
        print('  sample:', _txt[:120].replace(chr(10), ' '), '...')
        SMOKE_OK = True
    except Exception as _e:
        SMOKE_OK = False
        print('SMOKE TEST FAILED — do NOT submit the batch. Real error below:')
        print(' ', type(_e).__name__, ':', str(_e)[:400])
else:
    SMOKE_OK = True

## RUN 1 — submit

In [5]:
def sample_params():
    # Sonnet 5 rejects temperature / top_p / top_k. Diversity comes from the prompt: each card
    # samples 3-4 style hints out of 6 and carries different content.
    return {}

if not BATCH_ID and SMOKE_OK:
    random.seed(7)                        # different seed than NB2f so hint sampling differs
    requests, meta = [], {}
    for _, card in my_cards.iterrows():
        p = sample_params()
        meta[card['pair_id']] = p
        requests.append(Request(
            custom_id=card['pair_id'],
            params=MessageCreateParamsNonStreaming(
                model=MODEL_NAME,
                max_tokens=min(int(int(card['target_words']) * 5.5) + 1000, 120000),
                system=SYSTEM_PROMPT,
                messages=[{'role': 'user', 'content': build_prompt(card)}],
                thinking={'type': 'disabled'},
            ),
        ))
    batch = client.messages.batches.create(requests=requests)
    with open(BATCH_META, 'w', encoding='utf-8') as f:
        json.dump({'batch_id': batch.id, 'model': MODEL_NAME, 'n_sent': len(requests)},
                  f, ensure_ascii=False)
    print('=' * 60)
    print(f'SUBMITTED {len(requests)} retry requests | status: {batch.processing_status}')
    print(f'BATCH_ID = {batch.id}')
    print('=' * 60)
    print('Saved to', BATCH_META)
    print('NEXT: wait, then paste this id into BATCH_ID and re-run to COLLECT.')
elif not SMOKE_OK:
    print('smoke test failed — submit skipped. Fix params first.')
else:
    print('BATCH_ID is set — skipping submit, going to COLLECT.')

BATCH_ID is set — skipping submit, going to COLLECT.


## RUN 2 — collect + gate

Writes `aig_sonnet_retry.parquet`. Articles that pass here **supersede** their earlier reject: when
NB2j merges, for any duplicated `pair_id` it should keep the row with `gate_passed=True`. Cards that
fail again stay `gate_passed=False` and carry over to Gemini.

In [6]:
def make_record(card, text, gate, nwords, lok, error=''):
    return {
        'id': f'AI_{GEN_ID}_{card["pair_id"]}', 'text': text, 'label': 'ai', 'generator': GEN_ID,
        'source_pair_id': card['pair_id'],
        'temperature': 0, 'top_p': 0,          # Claude 5 exposes no sampling knobs
        'target_words': int(card['target_words']), 'actual_words': nwords,
        'entities_injected': card['entities_for_prompt'],
        'coverage_weighted': gate.get('weighted', 0), 'coverage_entities': gate.get('entity_cov', 0),
        'gate_passed': bool(gate.get('passed', False)), 'length_ok': bool(lok), 'error': error,
        'is_carry_over': card['pair_id'] in carry_over,
    }

if BATCH_ID:
    print(f'polling {BATCH_ID} ...', flush=True)
    while True:
        b = client.messages.batches.retrieve(BATCH_ID)
        if b.processing_status == 'ended':
            break
        print('   still processing:', b.request_counts, flush=True); time.sleep(60)
    print('ended:', b.request_counts, flush=True)

    card_by_id = {c['pair_id']: c for _, c in my_cards.iterrows()}
    records, n_ok, n_rej, n_err = [], 0, 0, 0
    tin_tot = tout_tot = 0

    for res in client.messages.batches.results(BATCH_ID):
        pid  = res.custom_id
        card = card_by_id.get(pid)
        if card is None:
            continue
        if res.result.type != 'succeeded':
            emsg = ''
            try:    emsg = str(res.result.error)[:200]
            except Exception: pass
            if n_err < 3:
                print(f'ERROR {pid}: {res.result.type} | {emsg}', flush=True)
            records.append(make_record(card, '', {}, 0, False,
                                       error=f'batch:{res.result.type}:{emsg[:100]}'))
            n_err += 1; continue
        msg = res.result.message
        tin_tot += msg.usage.input_tokens; tout_tot += msg.usage.output_tokens
        text = normalize_format(''.join(b.text for b in msg.content if b.type == 'text'))
        gate = acceptance_gate(text, card); lok, nwords = length_ok(text, int(card['target_words']))
        records.append(make_record(card, text, gate, nwords, lok))
        if gate['passed']: n_ok += 1
        else: n_rej += 1

    out  = pd.DataFrame(records)
    CKPT = f'{OUT_DIR}/aig_{GEN_ID}_retry.parquet'
    out.to_parquet(CKPT, index=False)

    cost = tin_tot/1e6*PRICE_IN*0.5 + tout_tot/1e6*PRICE_OUT*0.5
    print('=' * 60)
    print(f'processed {len(out)} | passed {n_ok} | failed {n_rej} | errored {n_err}')
    for grp, lbl in [(True, 'carry-over'), (False, 'fresh')]:
        sub = out[out['is_carry_over'] == grp]
        if len(sub):
            print(f'   {lbl:<11}: {len(sub):3d} sent | {int(sub["gate_passed"].sum()):3d} passed '
                  f'({100*sub["gate_passed"].mean():.0f}%)')
    print(f'cost (batch, 50%): ${cost:.2f}')
    n_md = int(out['text'].str.contains('**', regex=False).sum())
    n_nl = int(out['text'].str.contains(chr(10), regex=False).sum())
    print(f'markdown/newlines left: {n_md}/{n_nl} (must be 0)')
    print(f'saved {CKPT} | upload as aigt-aig-sonnet-retry')
    print('=' * 60)

polling msgbatch_013jEiEvWwxvPaCWSqJB7aGv ...
   still processing: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=485, succeeded=0)
   still processing: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=485, succeeded=0)
   still processing: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=485, succeeded=0)
   still processing: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=485, succeeded=0)
   still processing: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=485, succeeded=0)
   still processing: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=485, succeeded=0)
   still processing: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=485, succeeded=0)
   still processing: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=485, succeeded=0)
   still processing: MessageBatchRequestCounts(canceled=0, errored=0, expi

## Notes

- **Merge rule for NB2j:** concatenate all five outputs, then for any duplicated `source_pair_id`
  keep the row with `gate_passed=True`. A card salvaged here supersedes its earlier reject; one that
  fails again is still carried by the reject row and flows on to Gemini.
- **The relay is proven.** DeepSeek's 22 rejects contribute 0 cards to this batch because Qwen
  rescued every one of them. That's `failed - succeeded` doing its job across notebooks.
- **Deliberate quota drift.** Everything here is written *by Sonnet*, so its share grows while Opus
  stays at 380. The 450/450 split was a planning number, not a constraint. I'm spending Anthropic
  credit that can't be spent anywhere else, and Sonnet is the best gate-passer I have — so the
  trade is worth it. What I do watch is **generator diversity**, since my evaluation plan includes
  Leave-One-Generator-Out: at N_FRESH=350 Sonnet lands ~868, level with DeepSeek's 878, which keeps
  the corpus balanced rather than Sonnet-dominated.
- **Number emphasis is a fidelity fix, not detector tuning.** Both models failed the gate almost
  entirely on numbers (Sonnet rejects 45%, Opus 30%) while entities and agencies were fine — they
  paraphrase figures instead of carrying them. Asking for the numbers verbatim makes the AI article
  carry its card's facts, which is the entire basis of the human/AI pairing. None of my five
  detection features are touched.
- **Cost is measured, not guessed:** $6.45 / 470 = ~$0.0137 per article from my first Sonnet batch.